# Hospital data cleaning
Step-by-step version of `scripts/clean_data.py`. Run the cells from top to bottom.

In [1]:
import numpy as np
import pandas as pd

raw = pd.read_csv('../data/hospital_raw_data.csv')
print(raw.shape)
raw.head()

(55500, 17)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Department,Floor
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,Oncology,3
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,Endocrinology,2
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,Endocrinology,2
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,Endocrinology,4
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,Oncology,4


## 1. Look before cleaning
Names have random capital letters, there are duplicate rows and some billing amounts are negative.

In [2]:
print('duplicate rows:', raw.duplicated().sum())
print('negative billing:', (raw['Billing Amount'] < 0).sum())
print('missing cells:', raw.isna().sum().sum())
raw['Name'].head()

duplicate rows: 534
negative billing: 108
missing cells: 0


0    Bobby JacksOn
1     LesLie TErRy
2      DaNnY sMitH
3     andrEw waTtS
4    adrIENNE bEll
Name: Name, dtype: str

## 2. Fix text and remove duplicates

In [3]:
df = raw.copy()
for col in ['Name', 'Doctor', 'Hospital']:
    df[col] = df[col].str.strip().str.title()
df = df.drop_duplicates().reset_index(drop=True)
print('rows left:', len(df))

rows left: 54966


## 3. Dates and length of stay

In [4]:
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'])
df['los_days'] = (df['Discharge Date'] - df['Date of Admission']).dt.days
df['los_days'].describe()

count    54966.000000
mean        15.499290
std          8.661471
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: los_days, dtype: float64

## 4. Negative billing
A bill can't be negative. I replace those with the median bill of the same condition and admission type, and keep a flag column so the change is visible.

In [5]:
bad = df['Billing Amount'] < 0
df['billing_fixed'] = bad
df.loc[bad, 'Billing Amount'] = np.nan
df['Billing Amount'] = df['Billing Amount'].fillna(
    df.groupby(['Medical Condition', 'Admission Type'])['Billing Amount'].transform('median')).round(2)
print('fixed:', bad.sum())

fixed: 106


## 5. New columns

In [6]:
df['age_group'] = pd.cut(df['Age'], [0, 17, 34, 49, 64, 120], labels=['0-17', '18-34', '35-49', '50-64', '65+']).astype(str)
df['admit_year'] = df['Date of Admission'].dt.year
df['admit_month'] = df['Date of Admission'].dt.to_period('M').astype(str)
df['admit_weekday'] = df['Date of Admission'].dt.day_name()

## 6. 30-day readmission
No patient ID exists, so same name + gender + blood type = same patient. This is a limitation and is written in `docs/Methodology.md`.

In [7]:
key = ['Name', 'Gender', 'Blood Type']
df = df.sort_values(key + ['Date of Admission'])
gap = (df['Date of Admission'] - df.groupby(key)['Discharge Date'].shift()).dt.days
df['readmit_30d'] = gap.between(0, 30)
df = df.sort_index()
print('readmissions:', df['readmit_30d'].sum())

readmissions: 45


## 7. Tidy column names and save

In [8]:
df.columns = [c.lower().replace(' ', '_') for c in df.columns]
df = df.rename(columns={'date_of_admission': 'admission_date', 'name': 'patient_name', 'medical_condition': 'condition'})
df.to_csv('../data/hospital_cleaned_notebook.csv', index=False)
print('missing values left: %.3f%%' % (100 * df.isna().sum().sum() / df.size))

missing values left: 0.000%


The output is saved as `hospital_cleaned_notebook.csv` so it does not overwrite the file made by the script. The two files should be identical.